# Лабораторная работа №4. Акустическая модель FastSpeech 2

**Цель:** обучить акустическую модель для голоса RUSLAN, которая преобразует готовую последовательность фонем в mel-спектрограмму и одновременно учится предсказывать длительность, высоту тона (`F0`) и энергию.

Ноутбук подготовлен для последовательного ручного запуска. По умолчанию обучение выключено, а режим установлен в `smoke`. Сначала необходимо проверить все выводы короткого запуска и только затем переключаться на полный корпус.

## 1. Постановка задачи и метод

Вход модели — фонемы и паузы из лабораторной №3. Целевой объект — mel-спектрограмма аудиозаписи. FastSpeech 2 состоит из кодировщика фонем, предсказателей длительности, `F0` и энергии, регулятора длины и декодировщика спектрограммы.

```text
фонемы → encoder → duration / pitch / energy → length regulator → decoder → mel
```

Длительности берутся из MFA-выравнивания лабораторной №3. Поэтому внутренний aligner FastSpeech 2 отключён. Обрезка тишины также отключена: иначе mel перестанет совпадать с временными границами фонем. Токен `sil` остаётся частью входной последовательности.

## 2. Входные данные и признаки

Используются:

- `data/RUSLAN/RUSLAN/*.wav`: mono PCM16, 22 050 Гц;
- `data/lab3_g2p/utterance_phonemes.csv`: фонемная последовательность;
- `data/lab3_g2p/phoneme_alignments.csv`: границы фонем;
- 80-канальная mel-спектрограмма (`FFT=1024`, `win=1024`, `hop=256`);
- покадровые `F0` и энергия, вычисляемые Coqui TTS.

Запись `005321_RUSLAN` с приближённым выравниванием и записи длиннее 15 секунд исключаются автоматически и сохраняются в `excluded.csv` с причиной. Ограничение длины защищает 8 ГБ VRAM от выбросов до 50 секунд. Тестовая часть не передаётся обучающему коду.

In [1]:
from pathlib import Path
import json
import platform
import sys

# Работает как при запуске из корня, так и из каталога лабораторной.
def locate_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'lab_reqs.md').is_file():
            return candidate
    raise FileNotFoundError('Не найден корень репозитория')

REPO_ROOT = locate_repo_root()
LAB_DIR = REPO_ROOT / 'labs' / 'lab4_acoustic'
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from acoustic_data import (
    AcousticSettings, SplitSettings, choose_smoke_ids,
    create_or_load_splits, prepare_attention_masks, split_summary,
)

print('Корень:', REPO_ROOT)
print('Python:', sys.version.split()[0], '|', platform.platform())

Корень: C:\work projects\ITMO\tts_labs_itmo
Python: 3.12.1 | Windows-11-10.0.22631-SP0


## 3. Проверка окружения

Ожидается Python 3.11, PyTorch 2.5.1 с CUDA и курсовой форк Coqui TTS. В форке есть специальная токенизация многосимвольных MFA-фонем (`t̪s̪`, `rʲ`, `ɕː`). Обычная версия библиотеки может разбить их на отдельные Unicode-символы и не подходит.

In [2]:
try:
    import torch
    import TTS
    from TTS.tts.utils.text.characters import MulticharPhonemes
    from TTS.tts.utils.text.cleaners import multicharacter_phoneme_cleaners
    print('PyTorch:', torch.__version__)
    print('CUDA доступна:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM, GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
    print('Coqui TTS:', getattr(TTS, '__version__', 'version unavailable'))
    print('Поддержка MFA-токенов: OK')
except Exception as error:
    print('Окружение обучения пока не готово:', repr(error))
    print('Установите зависимости по инструкции labs/lab4_acoustic/README.md')

C:\work projects\ITMO\tts_labs_itmo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA доступна: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM, GiB: 8.0
Coqui TTS: 0.27.5
Поддержка MFA-токенов: OK


## 4. Фиксированное разделение корпуса

Разделение создаётся один раз: 90% `train`, 5% `validation`, 5% `test`. Используется `seed=42` и стратификация по длительности записи.

При повторном запуске блок загружает существующие CSV и проверяет fingerprint исходных данных, параметры, полное покрытие и отсутствие пересечений. Несовместимый кэш **не перезаписывается автоматически**. `force=True` следует включать только осознанно после анализа причины.

In [3]:
SPLIT_SETTINGS = SplitSettings(
    seed=42,
    validation_fraction=0.05,
    test_fraction=0.05,
    duration_bins=10,
    exclude_non_mfa=True,
    max_duration_s=15.0,
)
FORCE_RECREATE_SPLIT = False

splits, split_metadata = create_or_load_splits(
    REPO_ROOT, SPLIT_SETTINGS, force=FORCE_RECREATE_SPLIT
)
print('Статус кэша:', split_metadata['cache_status'])
print(json.dumps({k: v for k, v in split_metadata.items() if k != 'source_fingerprint'},
                 ensure_ascii=False, indent=2))
split_summary(splits).style.format({'hours': '{:.2f}', 'mean_duration_s': '{:.2f}', 'mean_phones': '{:.1f}'})

Статус кэша: loaded
{
  "schema_version": 2,
  "settings": {
    "seed": 42,
    "validation_fraction": 0.05,
    "test_fraction": 0.05,
    "duration_bins": 10,
    "exclude_non_mfa": true,
    "max_duration_s": 15.0
  },
  "lab3_source_fingerprint": "5146887bf32b4c772ad8848fa7bbb407a97f05870e0e038ee506b338b63aa0dd",
  "source_count": 21844,
  "counts": {
    "train": 19659,
    "validation": 1092,
    "test": 1093
  },
  "excluded_count": 256,
  "excluded_ids": [
    "001015_RUSLAN",
    "002155_RUSLAN",
    "002189_RUSLAN",
    "005321_RUSLAN",
    "006358_RUSLAN",
    "006388_RUSLAN",
    "008826_RUSLAN",
    "009636_RUSLAN",
    "012029_RUSLAN",
    "012058_RUSLAN",
    "012267_RUSLAN",
    "012548_RUSLAN",
    "016279_RUSLAN",
    "016577_RUSLAN",
    "017260_RUSLAN",
    "017964_RUSLAN",
    "018091_RUSLAN",
    "018095_RUSLAN",
    "018096_RUSLAN",
    "018106_RUSLAN",
    "018117_RUSLAN",
    "018118_RUSLAN",
    "018140_RUSLAN",
    "018143_RUSLAN",
    "018157_RUSLAN",
    "

,split,utterances,hours,mean_duration_s,mean_phones
0,train,19659,26.83,4.91,65.6
1,validation,1092,1.49,4.91,65.5
2,test,1093,1.49,4.90,65.3


In [4]:
train_ids = set(splits['train']['utterance_id'])
validation_ids = set(splits['validation']['utterance_id'])
test_ids = set(splits['test']['utterance_id'])
assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
assert '005321_RUSLAN' in split_metadata['excluded_ids']
assert splits['train']['duration_s'].max() <= SPLIT_SETTINGS.max_duration_s
assert splits['validation']['duration_s'].max() <= SPLIT_SETTINGS.max_duration_s
assert splits['test']['duration_s'].max() <= SPLIT_SETTINGS.max_duration_s
print('Утечки между частями отсутствуют.')
print('Train / validation / test:', len(train_ids), len(validation_ids), len(test_ids))
print('Исключено записей:', split_metadata['excluded_count'])
display(splits['train'].head(3))

Утечки между частями отсутствуют.
Train / validation / test: 19659 1092 1093
Исключено записей: 256


,utterance_id,audio_relpath,phones,phone_count,duration_s,alignment_method,split
0,000000_RUSLAN,data/RUSLAN/RUSLAN/000000_RUSLAN.wav,s̪ t̪ rʲ ɪ v o ʐ n̪ ɨ m tɕ u s̪ t̪ v ə m bʲ ɪ ...,30,2.244762,mfa,train
1,000001_RUSLAN,data/RUSLAN/RUSLAN/000001_RUSLAN.wav,k ɐ v o sil ɪ ɲ tʲ ɪ rʲ ɪ s̪ u j ʊ t̪ p rʲ ɪ z...,48,3.219909,mfa,train
2,000002_RUSLAN,data/RUSLAN/RUSLAN/000002_RUSLAN.wav,ʂ t̪ o p ə ʊ tɕ i tʲ ɪ ʎ n̪ ə v ə f j ɪ v o si...,29,2.334785,mfa,train


## 5. Перевод длительностей в mel-кадры

MFA хранит границы в секундах, а FastSpeech 2 работает с целым числом mel-кадров. Для каждой записи длительности округляются так, чтобы:

1. каждая фонема получила не менее одного кадра;
2. сумма длительностей точно совпала с числом кадров mel;
3. маска оставалась монотонной;
4. повторный запуск использовал уже созданные `.npy`.

Маски сохраняются как `uint8`, чтобы не расходовать несколько гигабайт без необходимости.

In [5]:
# Сначала обязательно 'smoke'. На 'full' переключаемся после анализа логов.
TRAINING_MODE = 'full'  # 'smoke' или 'full'
SMOKE_TRAIN_COUNT = 128
SMOKE_VALIDATION_COUNT = 32

if TRAINING_MODE == 'smoke':
    selected_ids = choose_smoke_ids(
        splits, train_count=SMOKE_TRAIN_COUNT,
        validation_count=SMOKE_VALIDATION_COUNT, seed=42
    )
elif TRAINING_MODE == 'full':
    selected_ids = None
else:
    raise ValueError("TRAINING_MODE должен быть 'smoke' или 'full'")

print('Режим:', TRAINING_MODE)
print('Записей для текущего запуска:', len(selected_ids) if selected_ids else 'весь train + validation')

Режим: full
Записей для текущего запуска: весь train + validation


In [6]:
attention_status = prepare_attention_masks(
    REPO_ROOT, splits, split_metadata, AcousticSettings(), only_ids=selected_ids
)
print(json.dumps(attention_status, ensure_ascii=False, indent=2))
if TRAINING_MODE == 'full':
    assert attention_status['complete'], 'Для полного обучения нужны маски всего train/validation/test'

{
  "fingerprint": "ad65f2a48b6a692f244a69f89fcea8d7f33f589c893fb74cf5c27bd3d32519ef",
  "complete": true,
  "generated_count": 21844,
  "expected_count": 21844,
  "settings": {
    "sample_rate": 22050,
    "fft_size": 1024,
    "win_length": 1024,
    "hop_length": 256,
    "num_mels": 80,
    "mel_fmin": 0.0,
    "mel_fmax": 8000.0,
    "center": true,
    "trim_silence": false
  },
  "built_now": 0,
  "reused_now": 21844,
  "metadata_path": "C:\\work projects\\ITMO\\tts_labs_itmo\\data\\lab4_acoustic\\metadata_attn_mask.txt"
}


In [7]:
import matplotlib.pyplot as plt
import numpy as np

example_id = sorted(selected_ids)[0] if selected_ids else splits['train'].iloc[0]['utterance_id']
mask_path = REPO_ROOT / 'data' / 'lab4_acoustic' / 'attention_masks' / f'{example_id}_attn.npy'
mask = np.load(mask_path)
assert mask.dtype == np.uint8 and np.all(mask.sum(axis=1) == 1)
plt.figure(figsize=(12, 4))
plt.imshow(mask.T, origin='lower', aspect='auto', interpolation='nearest')
plt.xlabel('Mel-кадр')
plt.ylabel('Индекс фонемы')
plt.title(f'Монотонная MFA-маска: {example_id}, shape={mask.shape}')
plt.colorbar(label='Принадлежность кадра')
plt.show()

C:\Users\mrkaz\AppData\Local\Temp\ipykernel_19672\719122734.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Инициализация FastSpeech 2

Ключевые решения конфигурации:

- `use_aligner=False`: используются длительности лабораторной №3;
- `use_pitch=True`, `use_energy=True`: это полный variance adaptor FastSpeech 2;
- `use_phonemes=False`: вход уже фонемный, повторный фонемизатор не нужен;
- `multicharacter_phoneme_cleaners`: пробелы разделяют цельные MFA-токены;
- `mixed_precision=True` при наличии CUDA: экономия памяти RTX 4060;
- `num_workers=0`: надёжный старт в Jupyter/Windows/WSL, позже можно увеличить;
- `start_by_longest=True`: ошибка нехватки VRAM проявится в самом начале.

Smoke test: 2 эпохи, batch size 4. Полный режим: 300 эпох максимум, batch size 8, не более двух периодических checkpoint-файлов.

In [8]:
from train_fastspeech2 import initialize_training

# continue_path восстанавливает не только веса, но и optimizer, scheduler, epoch и global step.
CONTINUE_PATH = str(REPO_ROOT / 'data' / 'lab4_acoustic' / 'runs' / 'ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2')
# Два процесса перекрывают загрузку WAV и mel-преобразование без перегрузки CPU на Windows.
NUM_WORKERS = 2

trainer, config, tokenizer, training_details = initialize_training(
    REPO_ROOT,
    splits,
    mode=TRAINING_MODE,
    selected_ids=selected_ids,
    num_workers=NUM_WORKERS,
    continue_path=CONTINUE_PATH,
)
print(json.dumps(training_details, ensure_ascii=False, indent=2))
print('Неизвестные токены:', tokenizer.not_found_characters)

C:\work projects\ITMO\tts_labs_itmo\.venv\Lib\site-packages\coqpit\coqpit.py:909: UserWarning: Type mismatch in Fastspeech2Config
Failed to deserialize field: characters (<class 'TTS.tts.configs.shared_configs.CharactersConfig'>) = {'characters_class': 'TTS.tts.utils.text.characters.MulticharPhonemes', 'vocab_dict': None, 'pad': '<PAD>', 'eos': None, 'bos': None, 'blank': None, 'characters': ['a', 'b', 'bʲ', 'bʲː', 'bː', 'c', 'cː', 'dʐː', 'dʲ', 'dʲː', 'd̪', 'd̪z̪', 'd̪ː', 'e', 'f', 'fʲ', 'fʲː', 'fː', 'i', 'j', 'jː', 'k', 'kː', 'm', 'mʲ', 'mʲː', 'mː', 'n̪', 'n̪ː', 'o', 'p', 'pʲ', 'pʲː', 'pː', 'r', 'rʲ', 'rʲː', 'rː', 'sil', 'sʲ', 'sʲː', 's̪', 's̪ː', 'tɕ', 'tɕː', 'tʂ', 'tʂː', 'tʲ', 'tʲː', 't̪', 't̪s̪', 't̪s̪ː', 't̪ː', 'u', 'v', 'vʲ', 'vʲː', 'vː', 'x', 'zʲ', 'z̪', 'z̪ː', 'æ', 'ç', 'ɐ', 'ɕː', 'ə', 'ɛ', 'ɟ', 'ɟː', 'ɡ', 'ɡː', 'ɣ', 'ɨ', 'ɪ', 'ɫ', 'ɫː', 'ɲ', 'ɲː', 'ɵ', 'ʂ', 'ʂː', 'ʉ', 'ʊ', 'ʎ', 'ʎː', 'ʐ', 'ʐː', 'ʑː'], 'punctuations': '', 'phonemes': None, 'is_unique': True, 'is_sorted': True}
V

{
  "mode": "full",
  "train_samples": 19659,
  "validation_samples": 1092,
  "vocabulary_size": 89,
  "device": "NVIDIA GeForce RTX 4060 Laptop GPU",
  "mixed_precision": true,
  "batch_size": 8,
  "epochs": 300,
  "num_workers": 2,
  "output_path": "C:\\work projects\\ITMO\\tts_labs_itmo\\data\\lab4_acoustic\\runs\\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2",
  "continue_path": "C:\\work projects\\ITMO\\tts_labs_itmo\\data\\lab4_acoustic\\runs\\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2",
  "resume_checkpoint": "C:/work projects/ITMO/tts_labs_itmo/data/lab4_acoustic/runs/ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2/checkpoint_715000.pth",
  "resume_epoch": 288,
  "resume_global_step": 715001,
  "quarantined_checkpoints": [],
  "course_tts_commit": "d8f03ab5f23411842cc429d0c93a506b772f5b1f"
}
Неизвестные токены: []


## 7. Запуск обучения

Ячейка ниже ничего не делает, пока `RUN_TRAINING=False`. Перед запуском проверьте:

- CUDA доступна и выбрана RTX 4060;
- токенизатор не сообщил неизвестных фонем;
- attention mask выглядит монотонной;
- выбран режим `smoke`;
- на диске достаточно места.

Для первого запуска измените только `RUN_TRAINING` на `True`.

In [9]:
RUN_TRAINING = False

if RUN_TRAINING:
    trainer.fit()
else:
    print('Обучение не запущено. После проверки установите RUN_TRAINING = True.')

 > Restoring best loss from best_model_707028.pth ...
 > Starting with loaded last best loss {'train_loss': 1.4498176712069524, 'eval_loss': 1.5369890795813665}

 > EPOCH: 288/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-22 22:56:05) 

   --> TIME: 2026-10-22 22:56:37 -- STEP: 49/2456 -- GLOBAL_STEP: 715050
     | > current_lr: 7.2249999999999994e-06  (7.225000000000006e-06)
     | > loss_spec: 1.5321643352508545  (1.3513962626457214)
     | > loss_dur: 0.054470907896757126  (0.05459290826503111)
     | > loss_pitch: 0.07777881622314453  (0.03280152524916492)
     | > loss_energy: 0.04706265404820442  (0.04329538550608012)
     | > loss: 1.7114766836166382  (1.4820860843269192)
     | > duration_error: 4.210742473602295  (4.291713349673212)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(9.6815, device='cuda:0')  (tensor(9.3970, device='cuda:0'))
     | > step_time: 0.


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.16440797205324528 (-0.07786496833518697)
     | > avg_loss_spec: 1.4369525419341191 (+0.04172173252812117)
     | > avg_loss_dur: 0.049519518362703144 (+0.0010429458485709323)
     | > avg_loss_pitch: 0.0422372764459363 (+0.0027871835700892036)
     | > avg_loss_energy: 0.04444401920945557 (-0.0001541738570840015)
     | > avg_loss: 1.5731533571525855 (+0.04539769446408304)
     | > avg_duration_error: 4.121777589232835 (+0.024120947166729856)


 > EPOCH: 290/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-22 23:19:11) 

   --> TIME: 2026-10-22 23:19:37 -- STEP: 35/2456 -- GLOBAL_STEP: 719950
     | > current_lr: 7.2750000000000015e-06  (7.275000000000002e-06)
     | > loss_spec: 0.9932126402854919  (1.2404522350856233)
     | > loss_dur: 0.032980140298604965  (0.052704776504210064)
     | > loss_pitch: 0.037891972810029984  (0.031728823


   --> TIME: 2026-10-22 23:20:06 -- STEP: 135/2456 -- GLOBAL_STEP: 720050
     | > current_lr: 7.2750000000000015e-06  (7.275000000000029e-06)
     | > loss_spec: 1.1205452680587769  (1.2640098183243358)
     | > loss_dur: 0.04128344729542732  (0.05050087896762071)
     | > loss_pitch: 0.014776313677430153  (0.03163027742670644)
     | > loss_energy: 0.03553396090865135  (0.04405554393651309)
     | > loss: 1.2121390104293823  (1.3901965154541867)
     | > duration_error: 3.8809702396392822  (4.149476791311193)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(14.3118, device='cuda:0')  (tensor(8.2031, device='cuda:0'))
     | > step_time: 0.1143  (0.11126112231501827)
     | > loader_time: 0.0246  (0.13706076939900708)


   --> TIME: 2026-10-22 23:20:21 -- STEP: 185/2456 -- GLOBAL_STEP: 720100
     | > current_lr: 7.2750000000000015e-06  (7.275000000000007e-06)
     | > loss_spec: 0.9233631491661072  (1.2677747781212263)
     | > loss_dur: 0.029800310730934143  (0.0499


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.22090602097687906 (+0.056498048923633776)
     | > avg_loss_spec: 1.4192347420586482 (-0.017717799875470952)
     | > avg_loss_dur: 0.049633443010626006 (+0.00011392464792286255)
     | > avg_loss_pitch: 0.04269663778444133 (+0.00045936133850502797)
     | > avg_loss_energy: 0.0447112119032277 (+0.00026719269377212945)
     | > avg_loss: 1.5562760313351949 (-0.016877325817390654)
     | > avg_duration_error: 4.121803862960253 (+2.627372741859091e-05)


 > EPOCH: 291/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-22 23:31:25) 

   --> TIME: 2026-10-22 23:31:50 -- STEP: 28/2456 -- GLOBAL_STEP: 722400
     | > current_lr: 7.300000000000001e-06  (7.300000000000007e-06)
     | > loss_spec: 1.2558859586715698  (1.2471144837992532)
     | > loss_dur: 0.04808090254664421  (0.045240863492446284)
     | > loss_pitch: 0.023377595469355583  (0.0452


   --> TIME: 2026-10-22 23:45:13 -- STEP: 221/2456 -- GLOBAL_STEP: 725050
     | > current_lr: 7.325e-06  (7.325000000000021e-06)
     | > loss_spec: 1.4822824001312256  (1.2735331681519066)
     | > loss_dur: 0.055103398859500885  (0.050404433059881175)
     | > loss_pitch: 0.03137475624680519  (0.034019333407350257)
     | > loss_energy: 0.05436253175139427  (0.043956236306965615)
     | > loss: 1.6231231689453125  (1.401913171709932)
     | > duration_error: 4.307976245880127  (4.1874870013327605)
     | > amp_scaler: 2048.0  (2048.0)
     | > grad_norm: tensor(10.7470, device='cuda:0')  (tensor(9.7100, device='cuda:0'))
     | > step_time: 0.1101  (0.11605310547945187)
     | > loader_time: 0.0313  (0.1731441085694602)


   --> TIME: 2026-10-22 23:45:26 -- STEP: 271/2456 -- GLOBAL_STEP: 725100
     | > current_lr: 7.325e-06  (7.32500000000003e-06)
     | > loss_spec: 1.0209383964538574  (1.2770910095904582)
     | > loss_dur: 0.03764600679278374  (0.051159606084942376)
     | > lo

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.18761564713937262 (-0.03407482041252988)
     | > avg_loss_spec: 1.3664664383287783 (-0.011394272027192454)
     | > avg_loss_dur: 0.049324157682281954 (-0.0009447921067476245)
     | > avg_loss_pitch: 0.03953586043445049 (+0.001376616026930226)
     | > avg_loss_energy: 0.04477545412050353 (-0.0003118367796694915)
     | > avg_loss: 1.5001018952440335 (-0.011274308628505647)
     | > avg_duration_error: 4.120941796126191 (-0.03452843560112484)

 > BEST MODEL : C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2\best_model_732200.pth

 > EPOCH: 295/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-23 00:20:42) 

   --> TIME: 2026-10-23 00:21:01 -- STEP: 0/2456 -- GLOBAL_STEP: 732200
     | > current_lr: 7.400000000000001e-06  (7.400000000000001e-06)
     | > loss_spec: 0.91072


   --> TIME: 2026-10-23 00:35:52 -- STEP: 393/2456 -- GLOBAL_STEP: 735050
     | > current_lr: 7.425e-06  (7.425000000000024e-06)
     | > loss_spec: 1.686167597770691  (1.2638021032622149)
     | > loss_dur: 0.08537085354328156  (0.04933607424716004)
     | > loss_pitch: 0.008428900502622128  (0.032285055275959144)
     | > loss_energy: 0.03765780106186867  (0.04394882488948395)
     | > loss: 1.8176252841949463  (1.3893720550088176)
     | > duration_error: 5.2085089683532715  (4.137015028460944)
     | > amp_scaler: 2048.0  (2048.0)
     | > grad_norm: tensor(10.6307, device='cuda:0')  (tensor(8.5952, device='cuda:0'))
     | > step_time: 0.0939  (0.12351455154613075)
     | > loader_time: 0.235  (0.1657013243997975)


   --> TIME: 2026-10-23 00:36:06 -- STEP: 443/2456 -- GLOBAL_STEP: 735100
     | > current_lr: 7.425e-06  (7.425000000000037e-06)
     | > loss_spec: 0.8717487454414368  (1.2656637368299095)
     | > loss_dur: 0.0448354072868824  (0.04951364642879495)
     | > loss_p


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.27404011620415586 (+0.1026300253691497)
     | > avg_loss_spec: 1.4024164968066746 (-0.04273655679490829)
     | > avg_loss_dur: 0.05185018038308179 (+0.001606352751453717)
     | > avg_loss_pitch: 0.0402458549235706 (+0.0010163664403888828)
     | > avg_loss_energy: 0.0443893296437131 (+0.00048818637927373204)
     | > avg_loss: 1.5389018606256555 (-0.03962564468383789)
     | > avg_duration_error: 4.206984452848079 (+0.04977815416123921)


 > EPOCH: 297/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-23 00:46:08) 

   --> TIME: 2026-10-23 00:46:35 -- STEP: 36/2456 -- GLOBAL_STEP: 737150
     | > current_lr: 7.45e-06  (7.449999999999996e-06)
     | > loss_spec: 0.8931045532226562  (1.2654883381393223)
     | > loss_dur: 0.04629773646593094  (0.05065170334031185)
     | > loss_pitch: 0.016319027170538902  (0.0383485917829805)
     | > lo


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.22050610471654816 (-0.0535340114876077)
     | > avg_loss_spec: 1.4043002446492516 (+0.0018837478425770549)
     | > avg_loss_dur: 0.04965725190661571 (-0.0021929284764660786)
     | > avg_loss_pitch: 0.04139862023293972 (+0.0011527653093691206)
     | > avg_loss_energy: 0.04430994841235654 (-7.938123135656228e-05)
     | > avg_loss: 1.5396660628142185 (+0.0007642021885629902)
     | > avg_duration_error: 4.131639335773606 (-0.07534511707447322)


 > EPOCH: 298/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-23 00:58:52) 

   --> TIME: 2026-10-23 00:59:17 -- STEP: 29/2456 -- GLOBAL_STEP: 739600
     | > current_lr: 7.475000000000001e-06  (7.475000000000004e-06)
     | > loss_spec: 1.3197637796401978  (1.2290642035418542)
     | > loss_dur: 0.051461536437273026  (0.04923212952141104)
     | > loss_pitch: 0.07219403237104416  (0.0328284459


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.1412186375370732 (-0.07928746717947496)
     | > avg_loss_spec: 1.3626502028218024 (-0.04165004182744925)
     | > avg_loss_dur: 0.049815559897709784 (+0.0001583079910940713)
     | > avg_loss_pitch: 0.04163365919418909 (+0.00023503896124937168)
     | > avg_loss_energy: 0.04456906260715591 (+0.00025911419479936887)
     | > avg_loss: 1.4986684768288223 (-0.04099758598539616)
     | > avg_duration_error: 4.139645350420913 (+0.008006014647307325)

 > BEST MODEL : C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2\best_model_742028.pth

 > EPOCH: 299/299
 --> C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2

 > TRAINING (2026-10-23 01:11:26) 

   --> TIME: 2026-10-23 01:11:51 -- STEP: 22/2456 -- GLOBAL_STEP: 742050
     | > current_lr: 7.5e-06  (7.5e-06)
     | > loss_spec: 1.0564994812011719  (1.1575214158

## 8. Что сохранить после запуска

Для анализа нужны:

1. полный вывод ячейки инициализации;
2. первые 20–30 шагов обучения;
3. итог каждой эпохи train/validation;
4. сообщения об ошибках или предупреждениях;
5. путь к каталогу запуска;
6. графики `loss`, `duration_error`, pitch и energy из TensorBoard.

TensorBoard запускается из корня репозитория:

```bash
tensorboard --logdir data/lab4_acoustic/runs
```

Тестовый набор пока не открывается: его используем один раз после выбора лучшего checkpoint.

In [10]:
runs_dir = REPO_ROOT / 'data' / 'lab4_acoustic' / 'runs'
if runs_dir.exists():
    run_dirs = sorted((path for path in runs_dir.iterdir() if path.is_dir()),
                      key=lambda path: path.stat().st_mtime, reverse=True)
    print('Последние каталоги запусков:')
    for path in run_dirs[:5]:
        checkpoints = sorted(path.glob('*.pth'))
        print('-', path, '| checkpoints:', len(checkpoints))
else:
    print('Каталог runs появится после первого запуска Trainer.')

Последние каталоги запусков:
- C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+42AM-9617ec2 | checkpoints: 4
- C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_full-October-18-2026_02+20PM-9617ec2 | checkpoints: 1
- C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_smoke-October-18-2026_02+20AM-9617ec2 | checkpoints: 2
- C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_smoke-October-18-2026_01+50AM-9617ec2 | checkpoints: 2
- C:\work projects\ITMO\tts_labs_itmo\data\lab4_acoustic\runs\ruslan_fastspeech2_smoke-October-18-2026_12+15AM-0000000 | checkpoints: 0
